# 01 Parsing the RG I XML File

This notebook parses the XML-encoded entries of *Repertorium Germanicum I*

Goals:

1. Load and inspect the XML file
2. Count the main XML elements
3. Extract the main texts
4. Export the extracted texts as a CSV file for further analysis

# 1 Setup

In [1]:
from pathlib import Path
from collections import Counter
import re
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import display

In [2]:
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

In [3]:
def find_project_root(start_path):
    """
    Search the current directory and its parents for data/raw/rg1.xml.
    """
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        xml_candidate = candidate / "data" / "raw" / "rg1.xml"

        if xml_candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data/raw/rg1.xml. "
        "Check that rg1.xml is stored in the project's data/raw directory."
    )


project_root = find_project_root(Path.cwd())

xml_path = project_root / "data" / "raw" / "rg1.xml"
output_dir = project_root / "data" / "derived"

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("XML path:", xml_path)
print("XML exists:", xml_path.exists())
print("Output directory:", output_dir)

Current working directory: c:\Digital Philology\measuring-manuscripts-project-template
Project root: C:\Digital Philology\measuring-manuscripts-project-template
XML path: C:\Digital Philology\measuring-manuscripts-project-template\data\raw\rg1.xml
XML exists: True
Output directory: C:\Digital Philology\measuring-manuscripts-project-template\data\derived


## 1. Loading and inspecting the XML

The XML is parsed as a hierarchical tree. Before extracting the text, the
root element, document volume, element counts, and available tag types are
inspected.

In [4]:
tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)
print("Root attributes:", root.attrib)
print("File size:", round(xml_path.stat().st_size / 1_000_000, 2), "MB")

Root tag: rg
Root attributes: {'vol': '1'}
File size: 2.4 MB


In [5]:
core_elements = [
    "lemma",
    "reg",
    "head",
    "sublemma",
    "date",
    "fund",
    "ka",
    "kz",
    "personenindex",
    "ortsindex"
]

core_element_counts = []

for tag in core_elements:
    count = len(root.findall(f".//{tag}"))

    core_element_counts.append({
        "element": tag,
        "count": count
    })

core_counts_df = pd.DataFrame(core_element_counts)

display(core_counts_df)

,element,count
0,lemma,3845
1,reg,3845
2,head,3845
3,sublemma,5589
4,date,1103
5,fund,5260
6,ka,732
7,kz,732
8,personenindex,3845
9,ortsindex,3845


In [6]:
tag_counts = Counter(
    element.tag
    for element in root.iter()
    if isinstance(element.tag, str)
)

tag_inventory_df = pd.DataFrame(
    tag_counts.items(),
    columns=["tag", "count"]
).sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

print("Number of distinct XML tag names:", len(tag_inventory_df))

display(tag_inventory_df.head(30))

Number of distinct XML tag names: 282


,tag,count
0,sublemma,5589
1,fund,5260
2,head,3845
3,reg,3845
4,lemma,3845
5,personenindex,3845
6,ortsindex,3845
7,abk296,3561
8,abk312,2959
9,abk671,1804


## 2. Text extraction rules

Two versions of every `<head>` and `<sublemma>` are created:

### `full_text`

Contains all visible textual content, including archival references.

### `analysis_text`

Preserves the original abbreviated text but removes content marked with
`<fund>`, because archival shelfmarks are metadata rather than linguistic
content.

The abbreviation elements themselves are removed as XML markup, but their
surface forms are retained.

In [7]:
def normalize_extracted_text(text):
    """
    Normalize whitespace and remove extraction artefacts around punctuation.
    """
    text = " ".join(text.split())

    # Avoid doubled full stops created when a <fund> element is removed
    # after an abbreviation that already ends in a full stop.
    text = re.sub(r"\.\s+\.", ".", text)

    # Remove whitespace before punctuation.
    text = re.sub(r"\s+([,;:!?])", r"\1", text)
    text = re.sub(r"\s+\.", ".", text)

    return text.strip()

In [8]:
def extract_text(element, skip_tags=None):
    """
    Recursively extract plain text from an XML element.

    Parameters
    ----------
    element:
        XML element from which text is extracted.

    skip_tags:
        Set of XML tag names whose content should be removed.
        For example: {"fund"}.

    Notes
    -----
    - Text inside abbreviation elements is preserved.
    - XML tags themselves are not included.
    - Empty <ka/> and <kz/> elements contribute no text.
    """
    if skip_tags is None:
        skip_tags = set()
    else:
        skip_tags = set(skip_tags)

    parts = []

    def walk(node):
        if node.text:
            parts.append(node.text)

        for child in node:
            if child.tag not in skip_tags:
                walk(child)

            # The tail occurs after the child element and must be preserved,
            # even when the child itself is skipped.
            if child.tail:
                parts.append(child.tail)

    walk(element)

    return normalize_extracted_text("".join(parts))

def clean_attribute(value):
    """
    Strip whitespace from an XML attribute.
    Return None for missing or empty values.
    """
    if value is None:
        return None

    value = value.strip()

    return value if value else None

first_lemma = root.find("./lemma")
first_head = first_lemma.find("./reg/head")
first_sublemma = first_lemma.find("./reg/sublemma")

print("HEAD")
print(extract_text(first_head))

print("\nFULL SUBLEMMA")
print(extract_text(first_sublemma))

print("\nSUBLEMMA WITHOUT <fund>")
print(extract_text(
    first_sublemma,
    skip_tags={"fund"}
))

HEAD
Abardus Alamannus

FULL SUBLEMMA
qui portavit litteras d. pape ex parte marchionis Moravie et revertitur ad ipsum cum litteris d. pape I 366 108v.

SUBLEMMA WITHOUT <fund>
qui portavit litteras d. pape ex parte marchionis Moravie et revertitur ad ipsum cum litteris d. pape.
